# Lab Exercise 10: Learning the XOR Boolean Function Using an MLP

This notebook aims to demonstrate how to implement a Multi-Layer Perceptron (MLP) to solve the non-linear XOR problem using two different deep learning libraries: Keras (TensorFlow high-level API) and TensorFlow's low-level API. We will define the dataset, build, compile, train, and evaluate the MLP for each implementation.

## 1. Create the Dataset

We will define the input (X) and output (y) arrays for all four XOR combinations.

In [1]:
import numpy as np

# Input data for XOR
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

# Output data for XOR
y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=np.float32)

print("Input (X):")
print(X)
print("\nOutput (y):")
print(y)

Input (X):
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]

Output (y):
[[0.]
 [1.]
 [1.]
 [0.]]


## Implementing with Keras (TensorFlow high-level API)

### 2. Build an MLP (Keras)

We will define a sequential model with an input layer, a hidden layer with ReLU activation, and an output layer with sigmoid activation.

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Build the Keras model
keras_model = keras.Sequential([
    layers.Input(shape=(2,)), # Input layer with 2 features
    layers.Dense(units=4, activation='relu', name='hidden_layer'), # Hidden layer with 4 neurons and ReLU activation
    layers.Dense(units=1, activation='sigmoid', name='output_layer') # Output layer with 1 neuron and sigmoid activation
])

keras_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer (Dense)            │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17 (68.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 0 (0.00 B)

### 3. Compile the Model (Keras)

We will compile the model using Binary Cross-Entropy loss and the Adam optimizer.

In [3]:
# Compile the Keras model
keras_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### 4. Train the Model (Keras)

Now, we will train the Keras model on the XOR dataset. We'll use a small number of epochs for demonstration, but you can experiment with more epochs, learning rates, and number of neurons to improve performance.

In [4]:
# Train the Keras model
history = keras_model.fit(X, y, epochs=1000, verbose=0) # verbose=0 to suppress output for each epoch

print("Training finished. Final accuracy:", history.history['accuracy'][-1])

Training finished. Final accuracy: 1.0


### 5. Evaluate the Model (Keras)

Finally, we will evaluate the trained Keras model by predicting outputs for all four input combinations and checking if it correctly learned the XOR function.

In [10]:
# Evaluate the Keras model
predictions_keras = (keras_model.predict(X) > 0.5).astype(int)

print("\nKeras Model Predictions for XOR:")
for i in range(len(X)):
    print(f"Input: {X[i]}, Expected Output: {int(y[i][0])}, Predicted Output: {predictions_keras[i][0]}")

keras_loss, keras_accuracy = keras_model.evaluate(X, y, verbose=0)
print(f"\nKeras Model Accuracy: {keras_accuracy:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

Keras Model Predictions for XOR:
Input: [0. 0.], Expected Output: 0, Predicted Output: 0
Input: [0. 1.], Expected Output: 1, Predicted Output: 1
Input: [1. 0.], Expected Output: 1, Predicted Output: 1
Input: [1. 1.], Expected Output: 0, Predicted Output: 0

Keras Model Accuracy: 1.0000


## Implementing with TensorFlow Low-Level API

### 2. Build an MLP (TensorFlow Low-Level)

Using TensorFlow's low-level API, we will manually define weights and biases for each layer and implement the forward pass.

In [6]:
tf.random.set_seed(42)

# Define weights and biases for the hidden layer
W1 = tf.Variable(tf.random.normal([2, 4]), name='W1') # 2 inputs, 4 hidden neurons
b1 = tf.Variable(tf.zeros([4]), name='b1')

# Define weights and biases for the output layer
W2 = tf.Variable(tf.random.normal([4, 1]), name='W2') # 4 hidden neurons, 1 output neuron
b2 = tf.Variable(tf.zeros([1]), name='b2')

# Define the forward pass function
def forward_pass(inputs):
    # Hidden layer: inputs * W1 + b1, then ReLU activation
    hidden_layer_output = tf.nn.relu(tf.matmul(inputs, W1) + b1)
    # Output layer: hidden_layer_output * W2 + b2, then Sigmoid activation
    output_layer_output = tf.nn.sigmoid(tf.matmul(hidden_layer_output, W2) + b2)
    return output_layer_output

### 3. Define Loss and Optimizer (TensorFlow Low-Level)

We will use `tf.keras.losses.BinaryCrossentropy` for the loss function and `tf.optimizers.Adam` for the optimizer. We'll also define a custom training step.

In [7]:
# Define the loss function
loss_fn = tf.keras.losses.BinaryCrossentropy()

# Define the optimizer
optimizer = tf.optimizers.Adam(learning_rate=0.01)

# Define a single training step
@tf.function
def train_step(inputs, targets):
    with tf.GradientTape() as tape:
        predictions = forward_pass(inputs)
        loss = loss_fn(targets, predictions)

    # Compute gradients
    gradients = tape.gradient(loss, [W1, b1, W2, b2])

    # Apply gradients to update weights and biases
    optimizer.apply_gradients(zip(gradients, [W1, b1, W2, b2]))
    return loss, predictions

### 4. Train the Model (TensorFlow Low-Level)

We will implement a training loop to iteratively update the model's weights and biases.

In [8]:
epochs_tf = 1000

print("Training TensorFlow Low-Level Model...")
for epoch in range(epochs_tf):
    loss_value, _ = train_step(X, y)
    # if epoch % 100 == 0:
    #     print(f"Epoch {epoch}, Loss: {loss_value.numpy():.4f}")

print("TensorFlow Low-Level Model Training finished.")

Training TensorFlow Low-Level Model...
TensorFlow Low-Level Model Training finished.


### 5. Evaluate the Model (TensorFlow Low-Level)

Finally, we will evaluate the trained TensorFlow low-level model by predicting outputs for all four input combinations.

In [11]:
# Evaluate the TensorFlow Low-Level model
predictions_tf_raw = forward_pass(X)
predictions_tf_raw_binary = (predictions_tf_raw > 0.5).numpy().astype(int)

print("\nTensorFlow Low-Level Model Predictions for XOR:")
correct_predictions = 0
for i in range(len(X)):
    print(f"Input: {X[i]}, Expected Output: {int(y[i][0])}, Predicted Output: {predictions_tf_raw_binary[i][0]}")
    if predictions_tf_raw_binary[i][0] == int(y[i][0]):
        correct_predictions += 1

tf_accuracy = correct_predictions / len(X)
print(f"\nTensorFlow Low-Level Model Accuracy: {tf_accuracy:.4f}")


TensorFlow Low-Level Model Predictions for XOR:
Input: [0. 0.], Expected Output: 0, Predicted Output: 0
Input: [0. 1.], Expected Output: 1, Predicted Output: 1
Input: [1. 0.], Expected Output: 1, Predicted Output: 1
Input: [1. 1.], Expected Output: 0, Predicted Output: 0

TensorFlow Low-Level Model Accuracy: 1.0000


## Conclusion

Both the Keras (high-level API) and TensorFlow (low-level API) implementations successfully learned the XOR function, demonstrating the flexibility and power of TensorFlow for building neural networks. You can experiment with hyperparameters like learning rate, activation functions (e.g., `tanh` instead of `relu`), and the number of neurons in the hidden layer to observe their impact on model performance.